# Session 7 - LangSmith & Observability

AI Agents & Workflows: Basics

This is the final session of the course.

In this notebook we will explore:

- why AI applications need observability
- what traces and runs are
- how to add local tracing for learning purposes
- how LangSmith tracing is enabled
- how to trace custom functions
- how to think about evaluation
- how to build simple test examples and evaluators
- how this connects to production AI applications

Main mental model:

```text
Build → Trace → Debug → Evaluate → Improve → Monitor
```

## 0. Setup

This notebook expects:

- `.env` file in the project root
- `OPENAI_API_KEY` inside `.env`
- optional `OPENAI_MODEL` inside `.env`

For LangSmith tracing, optionally add:

```bash
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=your_langsmith_api_key
LANGSMITH_PROJECT=ai-agents-workflows-basics
```

Required packages:

```bash
pip install langchain langchain-openai langsmith python-dotenv
```

The notebook is designed to still explain the concepts even if `LANGSMITH_API_KEY` is not configured.

In [ ]:
import os
import time
from functools import wraps
from dotenv import load_dotenv

# The notebook is inside notebooks/, so we load ../.env from the project root.
load_dotenv("../.env")

api_key = os.getenv("OPENAI_API_KEY")
model_name = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

langsmith_api_key = os.getenv("LANGSMITH_API_KEY")
langsmith_tracing = os.getenv("LANGSMITH_TRACING", "false")
langsmith_project = os.getenv("LANGSMITH_PROJECT", "ai-agents-workflows-basics")

print("OPENAI_API_KEY loaded:", bool(api_key))
print("Model:", model_name)
print("LANGSMITH_API_KEY loaded:", bool(langsmith_api_key))
print("LANGSMITH_TRACING:", langsmith_tracing)
print("LANGSMITH_PROJECT:", langsmith_project)

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=model_name,
    temperature=0
)

print("LLM initialized")

## 1. Why Observability?

A normal application can often be debugged with logs, exceptions, and metrics.

AI applications need more visibility because the behavior depends on:

- prompt
- model
- retrieved documents
- tool calls
- memory
- state
- user input
- model output

A final answer is not enough. We need to see how the answer was produced.

## 2. Black Box AI Call

Let's start with a simple model call.

This works, but we only see the final output.

In [ ]:
response = llm.invoke("Explain RAG in one short paragraph.")
print(response.content)

### Problem

We do not see:

- exact prompt structure
- timing
- model parameters
- token usage
- intermediate steps
- tool calls
- retrieved context

This is why observability matters.

## 3. Local Trace Decorator

Before using LangSmith, let's build a tiny local tracing decorator.

This is not a replacement for LangSmith.

It is just a teaching tool to understand what a trace captures.

In [ ]:
local_traces = []

def local_trace(name: str):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            start_time = time.time()
            trace_record = {
                "name": name,
                "inputs": {
                    "args": args,
                    "kwargs": kwargs,
                },
                "output": None,
                "error": None,
                "latency_seconds": None,
            }

            try:
                result = func(*args, **kwargs)
                trace_record["output"] = result
                return result
            except Exception as exc:
                trace_record["error"] = repr(exc)
                raise
            finally:
                trace_record["latency_seconds"] = round(time.time() - start_time, 4)
                local_traces.append(trace_record)

        return wrapper
    return decorator

## 4. Trace a Simple Function

Now we can trace a normal Python function.

In [ ]:
@local_trace("classify_ticket")
def classify_ticket(ticket: str) -> str:
    ticket_lower = ticket.lower()

    if "production" in ticket_lower or "down" in ticket_lower:
        return "incident"

    if "readme" in ticket_lower or "documentation" in ticket_lower:
        return "documentation"

    return "general"


category = classify_ticket("Production API is down after deployment.")
print(category)

In [ ]:
local_traces[-1]

## 5. Trace an LLM Function

Now let's trace a function that calls the model.

This gives us visibility into:

- input
- output
- latency
- errors

In [ ]:
@local_trace("explain_topic")
def explain_topic(topic: str, audience: str = "software engineers") -> str:
    prompt = f"""
Explain {topic} to {audience}.
Use maximum 5 bullet points.
Keep it practical.
"""
    response = llm.invoke(prompt)
    return response.content


answer = explain_topic("LangSmith observability")
print(answer)

In [ ]:
local_traces[-1]

## 6. Trace a Multi-Step Workflow

AI applications usually have multiple steps.

Example:

```text
classify ticket
  ↓
draft response
  ↓
final output
```

Let's trace each step.

In [ ]:
@local_trace("draft_response")
def draft_response(ticket: str, category: str) -> str:
    prompt = f"""
You are an internal engineering support assistant.

Ticket:
{ticket}

Category:
{category}

Draft a short response with next steps.
"""
    response = llm.invoke(prompt)
    return response.content


@local_trace("support_workflow")
def support_workflow(ticket: str) -> dict:
    category = classify_ticket(ticket)
    response = draft_response(ticket, category)

    return {
        "ticket": ticket,
        "category": category,
        "response": response,
    }


result = support_workflow("Production API is down after the latest deployment.")
result

In [ ]:
for trace in local_traces[-3:]:
    print(trace["name"], "=>", trace["latency_seconds"], "seconds")
    print("Error:", trace["error"])
    print("---")

## 7. Trace vs Run

In real observability systems:

```text
Trace = full request execution

Run = one step inside the trace
```

Example:

```text
Trace: support_workflow
  Run: classify_ticket
  Run: draft_response
    Run: LLM call
```

LangSmith gives a much better version of this than our simple local list.

## 8. LangSmith Setup

LangSmith tracing is usually enabled through environment variables.

In `.env`:

```bash
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=your_langsmith_api_key
LANGSMITH_PROJECT=ai-agents-workflows-basics
```

Keep the API key local. Do not commit it.

In [ ]:
# Make sure the current Python process has the values loaded from .env.
# If .env contains these values, LangSmith / LangChain can use them.

os.environ["LANGSMITH_PROJECT"] = langsmith_project

if langsmith_api_key:
    os.environ["LANGSMITH_API_KEY"] = langsmith_api_key

if langsmith_tracing:
    os.environ["LANGSMITH_TRACING"] = langsmith_tracing

print("LangSmith environment prepared")
print("Tracing enabled:", os.getenv("LANGSMITH_TRACING"))
print("Project:", os.getenv("LANGSMITH_PROJECT"))

## 9. Optional LangSmith Import

If the `langsmith` package is installed, we can use `@traceable`.

If not, install it from the project environment:

```bash
pip install langsmith
```

or add it to `requirements.txt`.

In [ ]:
try:
    from langsmith import traceable
    LANGSMITH_AVAILABLE = True
    print("LangSmith SDK available")
except ImportError:
    traceable = None
    LANGSMITH_AVAILABLE = False
    print("LangSmith SDK not installed. Add 'langsmith' to requirements.txt")

## 10. Trace Custom Functions with LangSmith

The `@traceable` decorator can trace custom Python functions.

This is useful when your application has steps that are not automatically traced by LangChain.

In [ ]:
if LANGSMITH_AVAILABLE:
    @traceable(name="langsmith_demo_classify_ticket")
    def langsmith_classify_ticket(ticket: str) -> str:
        ticket_lower = ticket.lower()

        if "production" in ticket_lower or "down" in ticket_lower:
            return "incident"

        if "readme" in ticket_lower or "documentation" in ticket_lower:
            return "documentation"

        return "general"

    print(langsmith_classify_ticket("Production API is down."))
else:
    print("Skipping LangSmith traceable demo because langsmith is not installed.")

## 11. Trace a LangChain Call

When LangSmith tracing is enabled, LangChain calls can be traced automatically.

Run this cell after setting:

```bash
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=...
LANGSMITH_PROJECT=...
```

Then open the LangSmith project UI.

In [ ]:
response = llm.invoke("Explain why AI agents need observability in 3 bullet points.")
print(response.content)

## 12. What To Inspect In A Trace

When reviewing a trace, look for:

- user input
- prompt
- model name
- model output
- tool calls
- retriever results
- latency
- token usage
- errors
- metadata
- tags

This helps identify the real failure point.

## 13. Debugging Example

Imagine this output is wrong:

```text
Production namespace does not require approval.
```

Possible failure points:

- wrong retrieved document
- missing production section
- bad prompt
- stale memory
- model ignored context
- source document is outdated

A trace helps us check each step.

## 14. Add Metadata and Tags

Metadata and tags make traces easier to filter.

Examples:

```text
course = ai-agents-workflows-basics
session = 7
component = support-assistant
environment = dev
version = v1
```

In production, this becomes very useful.

In [ ]:
# LangChain calls can receive config with tags and metadata.
# If LangSmith tracing is enabled, this information helps filter traces.

response = llm.invoke(
    "Explain tracing in one sentence.",
    config={
        "tags": ["session-7", "demo"],
        "metadata": {
            "course": "ai-agents-workflows-basics",
            "component": "observability-demo",
            "environment": "dev",
        }
    }
)

print(response.content)

## 15. Evaluation Basics

Tracing helps debug one run.

Evaluation helps test many runs.

Evaluation answers:

```text
Does my AI app work reliably across a dataset of examples?
```

This is similar to testing in software engineering.

## 16. Create A Small Evaluation Dataset

For the demo, we will create a small list of examples.

Each example has:

- input
- expected keywords

This is a simple but useful evaluator pattern.

In [ ]:
eval_examples = [
    {
        "input": "What is RAG?",
        "expected_keywords": ["retrieval", "context", "generation"],
    },
    {
        "input": "Why do agents need tools?",
        "expected_keywords": ["external", "actions"],
    },
    {
        "input": "What is LangGraph used for?",
        "expected_keywords": ["state", "workflow"],
    },
]

eval_examples

## 17. Define Target Application

This is the application we want to evaluate.

For simplicity, it is just an LLM call with a fixed system instruction.

In [ ]:
def course_assistant(question: str) -> str:
    prompt = f"""
You are an assistant for the course AI Agents & Workflows: Basics.

Answer the question in 3 short bullet points.

Question:
{question}
"""
    response = llm.invoke(prompt)
    return response.content

In [ ]:
print(course_assistant("What is RAG?"))

## 18. Simple Keyword Evaluator

This evaluator checks whether the answer contains expected keywords.

This is not perfect, but it is simple and transparent.

In [ ]:
def keyword_evaluator(answer: str, expected_keywords: list[str]) -> dict:
    answer_lower = answer.lower()
    found = []
    missing = []

    for keyword in expected_keywords:
        if keyword.lower() in answer_lower:
            found.append(keyword)
        else:
            missing.append(keyword)

    passed = len(missing) == 0

    return {
        "passed": passed,
        "found": found,
        "missing": missing,
        "score": len(found) / len(expected_keywords),
    }

## 19. Run Local Evaluation

Now evaluate the assistant on all examples.

In [ ]:
evaluation_results = []

for example in eval_examples:
    answer = course_assistant(example["input"])
    evaluation = keyword_evaluator(answer, example["expected_keywords"])

    evaluation_results.append({
        "input": example["input"],
        "answer": answer,
        "expected_keywords": example["expected_keywords"],
        "evaluation": evaluation,
    })

for result in evaluation_results:
    print("Input:", result["input"])
    print("Answer:", result["answer"])
    print("Evaluation:", result["evaluation"])
    print("---")

## 20. Summary Metrics

Even simple evaluation can give useful signals.

In [ ]:
scores = [result["evaluation"]["score"] for result in evaluation_results]
pass_count = sum(1 for result in evaluation_results if result["evaluation"]["passed"])

print("Examples:", len(evaluation_results))
print("Passed:", pass_count)
print("Average score:", round(sum(scores) / len(scores), 2))

## 21. Why Evaluation Matters

Without evaluation:

```text
We change a prompt and hope it still works.
```

With evaluation:

```text
We test the new prompt on known examples.
We compare results.
We catch regressions.
```

This is AI application testing.

## 22. Human Feedback

Human feedback is also part of observability.

Examples:

- thumbs up / thumbs down
- corrected answer
- reviewer comment
- issue category
- preferred response

Feedback can later become evaluation data.

In [ ]:
feedback_example = {
    "input": "How do I troubleshoot a Kubernetes workload?",
    "answer": "Check pod status and logs.",
    "user_feedback": "helpful",
    "reviewer_comment": "Good start, but should also mention events and probes.",
}

feedback_example

## 23. Production Monitoring

In production, monitor:

- latency
- error rate
- cost
- token usage
- failed tool calls
- retrieval quality
- model version
- prompt version
- user feedback
- evaluation score over time

AI apps need operational discipline.

## 24. Security and Privacy

Traces can contain sensitive data.

Be careful with:

- API keys
- secrets
- internal logs
- customer data
- personal information
- confidential documents

Do not blindly trace everything.

## 25. Redaction Example

Before logging or tracing, sensitive fields can be masked.

In [ ]:
def redact_text(text: str) -> str:
    # Very simple demo redaction.
    # Production redaction should be much more robust.
    replacements = {
        "OPENAI_API_KEY": "[REDACTED_KEY_NAME]",
        "password": "[REDACTED_PASSWORD]",
        "token": "[REDACTED_TOKEN]",
        "secret": "[REDACTED_SECRET]",
    }

    redacted = text

    for old, new in replacements.items():
        redacted = redacted.replace(old, new)

    return redacted


sample_log = "The deployment failed because password and token were missing."
print(redact_text(sample_log))

## 26. Final Mini Project

A good final project for this course:

```text
Internal Engineering Assistant
```

Features:

- uses LangChain model calls
- has tools
- uses short-term memory
- uses LangGraph routing
- searches documents with RAG
- traces runs with LangSmith
- evaluates answers on a small dataset

## 27. Course Wrap-Up

You now have the building blocks:

```text
LLM
Prompt
Tool
Agent
Memory
LangGraph
RAG
Observability
Evaluation
```

This is enough to start building useful internal AI agent workflows.

## 28. Discussion Questions

1. Why is final output not enough for debugging?
2. What is the difference between a trace and a run?
3. What should we inspect in a bad RAG answer?
4. Why do agents need tracing?
5. Why is evaluation important?
6. What should be redacted from traces?
7. What metrics matter in production AI systems?

## 29. Key Takeaways

- AI apps need observability.
- Traces show what happened during execution.
- Runs are individual steps inside traces.
- LangSmith helps inspect prompts, LLM calls, tools, retrievers, and errors.
- Evaluation tests the app across many examples.
- Human feedback is valuable evaluation data.
- Production AI needs monitoring, safety, and cost control.

## 30. Official Documentation

- LangSmith Observability: https://docs.langchain.com/langsmith/observability
- Tracing quickstart: https://docs.langchain.com/langsmith/observability-quickstart
- Trace LangChain applications: https://docs.langchain.com/langsmith/trace-with-langchain
- Evaluation: https://docs.langchain.com/langsmith/evaluate-llm-application
- Datasets: https://docs.langchain.com/langsmith/manage-datasets